# NGC 6302: a spectroscopist's walkthrough

NGC 6302, the Butterfly Nebula, is a bipolar planetary nebula whose far-infrared
spectrum is rich in crystalline dust. Kemper et al. (2002, A&A 394, 679) modelled
its *ISO* SWS/LWS spectrum with two dust shells, a cold one and a warm one, each
made of several species whose abundances and temperatures are free. What a
spectroscopist wants from such a model is two things: **the dust masses**, species
by species, and **whether the composition is unique**, that is, whether the
spectrum could be fitted equally well by a different mixture.

This notebook reproduces that model with ampere v2 on the 25 to 120 micron part of
the spectrum, and uses it to walk through what a fit with a *flexible likelihood*
(a Gaussian process, GP, on the residuals) gives you and how to read it. It is the
notebook form of `examples/ngc6302`, the v2 twin of the legacy `NGC6302.py`
(the migration guide's [NGC 6302 section](../migrating.rst) lists the mapping).
It assumes you know infrared dust spectroscopy and have never seen ampere. If you
want the idea behind the GP first, read [the concept page](../concept.rst); if you
want to see how the diagnostics are read on a toy problem and on a deliberately
wrong model, [Reading the diagnostics](../reading_the_diagnostics.rst) does both,
and this notebook is its second mild case: a real spectrum, a physical model, and a
GP that should absorb only what the model lacks.

**An honest word about the budget.** The model has eighteen free parameters and one
evaluation takes about 20 ms, so a documentation build can afford a few thousand
evaluations in total. A properly converged fit of this model takes tens of
thousands of steps and about an hour (the twin's record, quoted below). So the
fits here are *deliberately far too short*: they show every step of the workflow,
including the convergence verdict, and the numbers they print are those of a
chain that has barely left its starting point. The notebook says so wherever it
matters, and quotes the long runs' numbers, with their source, where they exist.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import scipy.stats as st

start = time.perf_counter()

# The twin lives in the repository's examples/ directory, not in the installed
# package: find the repository root from wherever the notebook is running.
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "examples" / "ngc6302").is_dir())
sys.path.insert(0, str(ROOT))

import astropy.units as u

from ampere.core import Spectrum
from ampere.inference import EmceeEngine
from ampere.results import (
    add_posterior_predictive,
    add_residuals,
    check_convergence,
    gp_localisation,
    gp_localisation_score,
    plot_anomaly_score,
    plot_gp_localisation,
    plot_posterior_predictive,
    plot_residuals,
    residual_whiteness,
    summary,
)
from examples.ngc6302 import dust_mass, generators, ngc6302

## Reading the data by hand

The spectrum is a two-column text file: wavelength in microns and flux density in
janskys. The twin ships it as `examples/NGC6302/NGC6302_100.tab` (the file the
legacy script reads). Reading it needs no ampere at all, and **these are the lines
you will replace with your own file**: read your wavelengths and fluxes however
you like, choose a wavelength range, and decide on an uncertainty. The twin takes
the 25 to 120 micron range and, since the file carries no errors, assigns each
point an uncertainty of 5 % of its flux, which is the legacy script's choice too.

What ampere wants is a `Spectrum`: a spectral axis with units, a flux with units,
and an uncertainty. Everything downstream (the model, the likelihood, the plots)
works from those.

In [ ]:
wavelength, flux = np.loadtxt(ROOT / "examples" / "NGC6302" / "NGC6302_100.tab", skiprows=2, unpack=True)
in_range = (wavelength >= 25.0) & (wavelength <= 120.0)
wavelength, flux = wavelength[in_range], flux[in_range]
observed = Spectrum(wavelength * u.um, flux * u.Jy, uncertainty=0.05 * flux * u.Jy)

print(f"{wavelength.size} points from {wavelength.min():.2f} to {wavelength.max():.2f} um")
print(f"flux: median {np.median(flux):.0f} Jy, {flux.min():.0f} to {flux.max():.0f} Jy")
print(f"spacing: median {np.median(np.diff(wavelength)):.3f} um")

The twin packages exactly this, plus a sort and a duplicate check, as
`load_observed_spectrum`. From here on we use that, and confirm it agrees with the
lines above.

In [ ]:
observed = generators.load_observed_spectrum()
print(np.allclose(observed.spectral_axis.values, wavelength), np.allclose(observed.values, flux))

## The model, species by species

A model in ampere is a class that **declares** its free parameters, each with a
prior (any frozen `scipy.stats` distribution), and its fixed inputs as *buffers*.
`KemperTwoShell` transcribes equations 6 and 7 of Kemper et al.: two shells of
dust, each a sum of species with a power-law density and a temperature gradient
between an outer (`0`) and an inner (`1`) temperature. The eight **opacity
tables** (laboratory data, one per species) are the model's buffers: loaded once,
not fitted. Each species' abundance is a free parameter in the shells where it
sits, as the base-10 logarithm of an abundance, with the same flat prior from
`-6` to `0` (legacy's own limits):

| species | cold shell | warm shell |
| --- | --- | --- |
| calcite | `logacold0` | |
| enstatite | `logacold1` | `logawarm1` |
| forsterite | `logacold2` | `logawarm2` |
| diopside | `logacold3` | |
| dolomite | `logacold4` | |
| iron | | `logawarm5` |
| water ice | `logacold6` | |
| (amorphous) olivine | `logacold7` | `logawarm7` |

That is eleven abundances. Four more parameters set the shells' temperatures:
`Tcold0` and `Twarm0` (the outer temperatures) and, in place of the inner
temperatures, two *fractions*.

**Why fractions, not temperatures.** Legacy fits an inner and an outer temperature
for each shell with the constraint that the inner one is hotter, which is a flat
prior over a triangle in the (outer, inner) plane. Many samplers handle a triangle
poorly, and a box with a rejection step wastes moves. The twin writes the inner
temperature as `Tcold1 = Tcold0 + Tcold_fraction * (80 - Tcold0)` (and likewise
for the warm shell, with 180 K as the ceiling), gives `Tcold0` a triangular
prior and the fraction a uniform one, and the induced joint prior on
(`Tcold0`, `Tcold1`) is **exactly** legacy's flat triangle. The Jacobian of the
change of variable cancels the triangle's density, as the twin's module docstring
proves in the section "Temperature ordering"; the inner temperatures survive as
derived quantities that the reporting functions compute for you.

The model needs no other knobs. Building it takes no arguments:

In [ ]:
model = ngc6302.build_model()
print(type(model).__name__, "with", len(generators.SPECIES), "opacity tables:", ", ".join(generators.SPECIES))

## The instrument and the two likelihoods

An `Instrument` is what turns the model's output into what the telescope would
have recorded. Here it has two steps: `Resample` evaluates the model on the
observed wavelengths, and `CalibrationScale` multiplies by a free overall
calibration factor. Legacy declared the calibration exact; the twin gives it a
5 % log-normal prior instead, because *ISO*'s absolute calibration is not that
good.

The **likelihood** says how the data scatter around the model. Two are built
here, on the same model, instrument and data:

* **Independent noise** (`IndependentNoise`): every point scatters on its own by
  its quoted 5 %. This is the usual least-squares assumption, and it is
  confident: if the model leaves *structure* in the residuals, this likelihood
  treats the structure as noise and reports parameters tighter than the data
  justify.
* **A Gaussian process** (`GaussianProcessNoise` with a `Matern32` kernel and the
  `QuasisepGP` solver, which is exact for this kernel on a sorted grid and costs
  order N, not N cubed). The residuals are allowed a correlated component of
  amplitude *a* (in Jy) and length scale *l* (in microns), both fitted. If the
  model is right the amplitude goes to zero and the two likelihoods agree; if it
  is wrong, the GP carries the part that is wrong and the model's parameters are
  protected from it.

The GP's priors are the one place where *you* decide how much of the data the
GP may take, and [the diagnostics page](../reading_the_diagnostics.rst) recommends
choosing them against the spectrum itself. Here the twin keeps legacy's own
length-scale prior, a half-normal of scale 0.1 micron, and an amplitude prior
that is a half-normal of scale 100 Jy. Set against this spectrum those say:

In [ ]:
step = np.median(np.diff(wavelength))
print(f"amplitude prior scale 100 Jy = {100 / np.median(flux):.0%} of the median flux ({np.median(flux):.0f} Jy)")
print(f"length-scale prior scale 0.1 um = {0.1 / step:.2f} x the median sampling interval ({step:.3f} um)")
print(f"the spectrum spans {np.ptp(wavelength):.0f} um")

Read against the page's "Choosing the kernel's priors": the amplitude prior lets
the GP carry about an eighth of the flux, which is permissive enough to matter
(the gross case's prior was about as large as the flux itself, and the GP
took a third of it); and the length-scale prior sits *at the sampling interval*,
so the GP can only be correlated between neighbouring pixels, structure on the
scale of an instrumental resolution element and below, not the broad dust bands.
That is the legacy script's choice, kept here so that the twin stays a twin. It
means that this GP is a tool for narrow features the model cannot make, and any
broad mismatch will have to show up in the model's parameters instead.

Now we build the two problems. A `FittingProblem` is the model, the datasets
(each an observation, its instrument and its likelihood) and a seed. The twin's
`build_problem` composes them; `gp=False` swaps in independent noise.

In [ ]:
problem_independent = ngc6302.build_problem(gp=False)
problem_flexible = ngc6302.build_problem(gp=True)

print(len(problem_independent.parameters.free_labels()), "free parameters under independent noise,",
      len(problem_flexible.parameters.free_labels()), "with the GP:")
print(", ".join(problem_flexible.parameters.free_labels()))

Eighteen free parameters: eleven abundances, four temperature parameters, the
calibration factor and the GP's two hyperparameters (the independent problem has
the first sixteen). Everything else, including the opacities and the 5 %, is
fixed.

## Where to start

An ensemble sampler (emcee) needs a *start*: one point per walker. By default,
ampere (since W7.12) first runs an optimiser, finds the posterior's mode and starts
the walkers in a ball around it, because a start from prior draws in eighteen
dimensions takes far more steps than any short run has. On this problem that
optimiser costs about four minutes (a single start of the scipy optimiser),
which is more than this notebook's whole budget. (If the optimiser's mode sat on a
bound, as a GP amplitude driven to zero would, the *mixed* start would draw that
coordinate from the prior and start the others at the mode; the run records which
start it used in `ampere_start_kind`.) So we do not use the default, and **say so**: a
start is a declaration you control, and the spectroscopist's natural one is the
published solution.

Kemper et al.'s solution is recorded in the twin as `generators.TRUTH` (the
eleven log-abundances, `Tcold0`, `Twarm0` and the two fractions that recover the
published inner temperatures), the calibration factor starts at one, and the
GP starts at an amplitude of 3 Jy (a percent or so of the flux) and a length
scale of 0.1 micron. We build the start as a `(walkers, parameters)` array: the
centre plus 1 % Gaussian jitter in each coordinate (and a floor of 0.001), and
check that the centre, and every row, has a finite log posterior. emcee needs
more than twice as many walkers as parameters, so 38.

In [ ]:
WALKERS = 38


def paper_start(problem, flexible, seed=1):
    centre = {f"model.{name}": value for name, value in generators.TRUTH.items()}
    centre["iso.instrument.calibration_scale.scale"] = 1.0
    if flexible:
        centre["iso.likelihood.amplitude"] = 3.0
        centre["iso.likelihood.length_scale"] = 0.1
    x = problem.parameters.pack(centre)
    jitter = (0.01 * np.abs(x) + 1e-3) * np.random.default_rng(seed).standard_normal((WALKERS, x.size))
    return x, x + jitter


centre_flexible, start_flexible = paper_start(problem_flexible, flexible=True)
centre_independent, start_independent = paper_start(problem_independent, flexible=False)

print(f"log posterior at the published solution: {problem_independent.log_prob(centre_independent):.1f} (independent), "
      f"{problem_flexible.log_prob(centre_flexible):.1f} (with the GP)")
assert all(np.isfinite(problem_flexible.log_prob(row)) for row in start_flexible)
assert all(np.isfinite(problem_independent.log_prob(row)) for row in start_independent)

## Two short fits

Each fit is an `EmceeEngine` bound to a problem; `run(steps, burn_in=, initial=)`
returns the **run**, an ArviZ `DataTree` holding the draws, the data and the
provenance. Passing `initial=` records `ampere_start_kind = "supplied"`, which
the second line of each output below prints. The budgets are chosen so that the
whole notebook runs in about a minute: 38 walkers for 40 steps (the
first 20 discarded) with independent noise, and 70 steps (the first 35 discarded)
with the GP.

In [ ]:
BUDGET_INDEPENDENT = (40, 20)  # steps, burn-in
BUDGET_FLEXIBLE = (70, 35)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the ensemble-size note and arviz's short-chain note
    t0 = time.perf_counter()
    run_independent = EmceeEngine(problem_independent, walkers=WALKERS).run(
        BUDGET_INDEPENDENT[0], burn_in=BUDGET_INDEPENDENT[1], initial=start_independent
    )
    time_independent = time.perf_counter() - t0

print(f"emcee, independent noise: {time_independent:.1f} s; start kind {run_independent.attrs['ampere_start_kind']!r}")

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    t0 = time.perf_counter()
    run_flexible = EmceeEngine(problem_flexible, walkers=WALKERS).run(
        BUDGET_FLEXIBLE[0], burn_in=BUDGET_FLEXIBLE[1], initial=start_flexible
    )
    time_flexible = time.perf_counter() - t0

print(f"emcee, with the GP: {time_flexible:.1f} s; start kind {run_flexible.attrs['ampere_start_kind']!r}")

### The convergence verdict

`summary` is ArviZ's table of the posterior, and `check_convergence` is ampere's
reading of it: R-hat (below 1.1) and bulk effective sample size (at least 100),
with a remedy in words. `summary` also *warns* when a chain fails, and the warning
below is that. We let it speak rather than silence it.

In [ ]:
verdict = check_convergence(run_flexible)
print(verdict)
summary(run_flexible)

**Not converged, as expected.** With 35 retained steps per walker the ensemble has
not had time to explore; effective sample sizes in the fifties are what 38 walkers
that have only just begun to decorrelate give (about one per walker), and R-hat is
large because the walkers' short chains have not forgotten their jittered starts.
The means and widths in this table describe a ball around the published solution,
not the posterior. Do not quote them.

How long would it take? The twin's record (the module docstring of
`examples/ngc6302/ngc6302.py`, lines 148 to 215, run on 2026-09-29) fitted *synthetic*
data made from the same published solution with exactly this model: 50 walkers for
10 000 steps with the GP, started from the prior (500 000 evaluations, 79 minutes).
Its R-hat was still between 1.69 and 2.90 and its bulk ESS between 57 and 78 on all
eighteen parameters (lines 152 to 175). The walkers had not mixed after four hundred
times the steps used here, and the record traces the reason to the posterior itself,
which the last section shows: more steps at the same move were "the wrong lever".
So the verdict is not only a statement about this notebook's budget. Keep that in
mind as we read the diagnostics: they are the tools you would use on a converged
chain, and a short chain makes their answers less trustworthy, not different in kind.

## Every diagnostic, on this fit

[Reading the diagnostics](../reading_the_diagnostics.rst) describes four plots and
the one sentence each can support. We apply them in turn. The diagnostics that
need the posterior draws (residuals, replicates, the conditioned GP) are
evaluated on a thinned subset, about 30 draws, so that they cost seconds.

### The residual plot and its whiteness test

This plot is for fits with **independent** noise (a GP's residuals are whitened by
construction), so it goes on the first fit. It draws the standardised residuals,
(data minus model) over uncertainty, against wavelength, and below them the
autocorrelation of the residuals binned by separation, with the permutation test
of whether the residuals are white.

In [ ]:
def thin_for(run, target=30):
    n_draws = run["posterior"].sizes["chain"] * run["posterior"].sizes["draw"]
    return max(1, n_draws // target)


residuals = add_residuals(run_independent, problem_independent, thin=thin_for(run_independent))
whiteness = residual_whiteness(residuals)
print(whiteness)
plot_residuals(residuals);

**What it shows.** The residuals are not noise. The standardised residuals swing
between about +4.5 and -3.5 standard deviations, smoothly, over several microns:
positive from 28 to 36 micron, negative at 25 to 27 micron and again at 44 to 48
micron, with smaller excursions near 40, 69 and 90 to 95 micron. The whiteness
test confirms it: the statistic is of order 9 000 and *p* sits at the test's floor of
0.005 (199 permutations), and the autocorrelation is positive at every
separation up to a micron. Those are the regions of the crystalline silicates and
carbonates in the model (forsterite has bands near 33.6 and 69 micron, calcite
near 28, 44 and 92 micron), but the coincidence is only suggestive: the plot says
*where*, not *why*.

*It supports:* these residuals are not consistent with independent noise of 5 %, and
the model, the uncertainties or both leave structure in this spectrum, mostly at
25 to 50 micron. *It does not support:* what the structure is, or that any species
is mis-assigned. Remember what the chain is: it has stayed in the neighbourhood of
the published solution, the temperatures within a few kelvin and the abundances
within a few tenths of a dex (see the dust-mass table below), so
these are close to the residuals of the published composition, **not** of a
converged best fit. A converged fit would reduce them, and the plot would then say
how much is left. **The decision** it prompts is the page's: either fix the model
(more species, a grain-shape distribution, an emission-line component) or let a GP
take the rest, which is the next fit.

### The GP localisation, and the GP's amplitude

With the flexible likelihood the question changes: the GP has absorbed whatever it
could, so the interesting thing is *where*. The plot draws the **conditioned GP
mean**, what the GP says the model is missing at each wavelength given the data,
with its 68 % band. (The caveat printed under the axis is the plot's own, and the
page reads it clause by clause.)

In [ ]:
localisation = gp_localisation(run_flexible, problem_flexible, thin=thin_for(run_flexible))
plot_gp_localisation(localisation);

**What it shows.** The conditioned mean is non-zero in the same places the residuals
were: about +3.5 Jy at 29 to 33 micron, about -3 Jy at 25 to 27 micron and
about -1 Jy at 44 to 47 micron, and it is flat and consistent with zero from 55
micron to the red end. So the GP says the model lacks something at 25 to 50 micron,
in the crystalline-silicate complex, and nowhere else. *It supports:* the GP
absorbed structure of a few janskys at those wavelengths. *It does not support:*
"the model is wrong" in general, nor a species. Only the 29 to 33 micron
excursion has a band that excludes zero, and not by much.

Now the number the page tells you to report next to the result: **the GP's
amplitude, as a fraction of the flux**.

In [ ]:
amplitude = np.asarray(run_flexible["posterior"].dataset["iso.likelihood.amplitude"]).ravel()
length = np.asarray(run_flexible["posterior"].dataset["iso.likelihood.length_scale"]).ravel()
flux_60 = flux[np.argmin(np.abs(wavelength - 60.0))]
print(f"GP amplitude: median {np.median(amplitude):.2f} Jy ({np.median(amplitude) / flux_60:.2%} of the {flux_60:.0f} Jy at 60 um, "
      f"{np.median(amplitude) / np.median(flux):.2%} of the median flux); prior scale 100 Jy")
print(f"GP length scale: median {np.median(length):.3f} um (the sampling interval is {step:.3f} um)")

**The mild case, against the gross one.** The page opened with a power law fitted to a
quasar's silicate emission, where the GP took about a third of the flux. Here the
GP's amplitude is about 2.9 Jy, **0.3 % of the flux at 60 micron**, a hundredth
of the gross case's share, at a length scale of 0.095 micron, a fraction of the
sampling interval. The model is not being replaced by the GP.

But read it for what it is. The amplitude started at 3 Jy and has not moved: with
seventy steps the chain cannot say whether the data want 3 Jy or 30. The size of the
mismatch the residual plot shows (several standard deviations of 5 % of 600 Jy is of
the order of 100 Jy) is **far** larger than a GP of 3 Jy, and a GP whose length
scale is below the pixel spacing could not carry a five-micron feature however
large its amplitude: it is correlated only between neighbouring pixels. So what the
GP reports here is mostly its start and its prior, and the honest sentence is that
the mismatch lives in the model, in structure much broader than the GP's length
scale. The twin's record has two numbers for the amplitude, both on *synthetic*
data made from the published solution (module docstring, lines 148 to 215 and 280
to 300): the unconverged emcee run's mean of 37 Jy (standard deviation 61 Jy), about
4 % of the flux at 60 micron, from walkers in a low-posterior region where the GP
was absorbing residuals; and, at the truth with the nuisance parameters profiled,
an amplitude driven to zero, as data without misspecification should give. Neither
is a measurement for the real spectrum, whose converged amplitude is not in the
record. It is what a successor run should report.

### The posterior-predictive check

`plot_posterior_predictive` asks: if the fitted model were true, would data like
mine be typical? It simulates a replicate data set from each retained draw, with
the likelihood's own noise, and compares a discrepancy statistic (by default the
summed squared standardised residuals) of the replicates with the observed one.
It goes on the independent fit, where it is most informative.

In [ ]:
ppc = add_posterior_predictive(run_independent, problem_independent, thin=thin_for(run_independent))
figure = plot_posterior_predictive(ppc)

**What it shows.** The left panel's replicate band follows the broad shape of the
spectrum but misses the structure at 28 to 50 micron, where the observations leave
the band. On the right, the replicates' statistic piles up near 600 to 700 (about
the number of points, 625, as it should for noise alone), and the observed value,
about 1 500, lies far to the right of every replicate: the *p*-value is zero, at the
resolution of 38 replicates. *It supports:* the model, with the noise you gave it,
does not reproduce this spectrum as a whole. *It does not support:* where or why;
that is what the previous plots were for.

### The anomaly score

`plot_anomaly_score` shows the conditioned GP mean in units of its own uncertainty,
one number per wavelength, so that different fits and data sets can be compared.

In [ ]:
score = gp_localisation_score(localisation)
plot_anomaly_score(score);

**What it shows.** The score peaks at about 1.3 near 26 micron and again near 30
micron, with a secondary peak near 45 micron and a small one near 113 micron (where the residual
plot shows a single-point step), and is below 0.1 over most of the rest.
*It supports:* where, relative to its own uncertainty, the GP is working hardest,
and the ranking is the localisation's. *It does not support:* a significance: the
page warns that the score ranks places within one fit and does not measure how
large a deficiency is. A maximum of 1.3 says that the GP's correction is barely
above its own uncertainty even at the worst place, which is a statement about the
GP's priors and start, not about the spectrum. **The decision** is the page's: use
it to decide where to look, and here that is 25 to 50 micron.

## The dust masses

The product a spectroscopist wants is not the parameter table but the **dust
mass** in each species and each shell. `examples/ngc6302/dust_mass.py` carries
equations 4 and 5 of Kemper et al. from the legacy `NGC6302-calculate-dust-mass.py`
over the posterior *draws*: each draw's abundances and temperatures give a
mass per species and shell, and the table reports the 16th, 50th and 84th
percentiles over the draws. The shell's overall size is anchored to the
published amorphous-olivine solution, as in the legacy script. Beside it, the
same function at the published solution itself (which reproduces the legacy
script's printed numbers, as the twin's tests check):

In [ ]:
table = dust_mass.dust_masses(run_flexible)
published = dust_mass.dust_masses_at({**generators.TRUTH, **ngc6302.derived_temperatures(generators.TRUTH)})

print(f"{'species':<10} {'shell':<5} {'this fit: 16% / median / 84%  (Msun)':<42} published")
for shell in ("cold", "warm"):
    for name in dust_mass.SPECIES_NAMES:
        q = table[shell][name]
        print(f"{name:<10} {shell:<5} {q['q0.16']:11.3g} {q['q0.50']:11.3g} {q['q0.84']:11.3g}      {published[shell][name]:11.3g}")
for shell in ("cold", "warm"):
    q = table["totals"][shell]
    print(f"{'total':<10} {shell:<5} {q['q0.16']:11.3g} {q['q0.50']:11.3g} {q['q0.84']:11.3g}      {published['totals'][shell]:11.3g}")

**Reading it.** The cold shell's total is about 0.05 solar masses in both
columns, dominated by amorphous olivine, because the published solution anchors the
scale and the chain has not left its neighbourhood. The warm shell's total is of the
order of 10^-5 solar masses and is carried by iron and olivine. Where the fit and
the published solution differ, they differ by factors of two or so in the minor
species: cold forsterite, diopside and ice, and warm iron. These are the
abundances that **drifted** in the chain (each by a few tenths of a dex, a couple of
posterior widths, in a direction the data pull in), and a chain with R-hat above 2
cannot say whether they will settle there. The 16 to 84 % ranges are the spread of
a short, unconverged ensemble and are narrower than any honest uncertainty. So the
table shows the *form* of the post-fit product, a posterior over masses propagated
draw by draw through the physics, and not yet its content. (The published solution
is `dust_masses_at(TRUTH)`, the 2002 solution as the legacy script hard-codes it;
Kemper et al.'s own table is the comparison for a converged run, and no numbers
from it are quoted here.)

## The second mode: is the composition unique?

The twin's record answers the second question, and the answer is that **the
data do not say**. After the emcee run above failed to converge, the twin's
module docstring (lines 236 to 320, dated 2026-09-30) records four further runs
on the same synthetic spectrum (made from the published solution, so the truth
is known): zeus from the prior, zeus from an optimiser's mode, zeus from the
best walker of the first, and the nested sampler nautilus. Taken together they
find **two distinct modes of the same posterior height**:

| | `Tcold0` (K) | `logacold7` (olivine, cold) | log posterior of the best draw | found by |
| --- | --- | --- | --- | --- |
| mode A (contains the truth, 36.1 K and -1.08) | about 33.6 +- 1.0 | about -1.27 +- 0.09 | -3098 | nautilus |
| mode B | about 40.5 | about -0.45 +- 0.16 | -3099 | zeus, started from its best walker |

(Figures from the docstring's runs 3 and 4 and its summary, lines 261 to 300.) The
two are 3.7 posterior widths apart at their closest high-posterior draws, and the
straight line between them drops 230 log-units: distinct modes, not one curved
ridge. In the docstring's words, "two dust compositions fit the 5 %-uncertainty
spectrum equally well". Each sampler found one of them, and the emcee coverage run
above never reached either: its wide intervals were the GP absorbing residuals at
an amplitude of 37 Jy, in a region some ninety log-units below the modes. Its
coverage of the truth "should not be read as evidence that the run sampled the
posterior".

**What this means for you.** An ensemble sampler started in one mode reports only
that mode, and its R-hat can look fine once it has settled there. This is
the open question of the legacy backlog. What would settle it is recorded as well: a nested
sampler whose bound is built to hold several modes (dynesty's multi-ellipsoid
decomposition, or nautilus with more live points), run once, with the mass of each
mode reported; or, on the science side, information that the spectrum alone does
not carry, such as the far-infrared photometry the script leaves out, or a prior
on which species are present. Until then, the fit above is a statement about the
neighbourhood of the published solution, which is exactly what a start there
buys.

In [ ]:
elapsed = time.perf_counter() - start
print(
    f"whole notebook: {elapsed:.1f} s (independent fit: {WALKERS} walkers x {BUDGET_INDEPENDENT[0]} steps, "
    f"burn-in {BUDGET_INDEPENDENT[1]}, {time_independent:.1f} s; GP fit: {WALKERS} x {BUDGET_FLEXIBLE[0]}, "
    f"burn-in {BUDGET_FLEXIBLE[1]}, {time_flexible:.1f} s)"
)

**Budget and wall time.** The independent fit used 38 walkers for 40 steps (first
20 discarded: 1 520 model evaluations) and the GP fit 38 walkers for 70 steps (first
35 discarded: 2 660 evaluations); the cell above prints the measured times, which
are of the order of a minute for the whole notebook in the `dev` pixi
environment. Neither fit converged and neither was meant to. The long-run numbers
quoted in the text (R-hat 1.69 to 2.90 after 79 minutes, the two modes) are not
produced here: they are the twin's module docstring (`examples/ngc6302/ngc6302.py`,
runs dated 2026-09-29 and 2026-09-30), recorded once and kept outside git.